# Findings Register
**66 STRUCTURED FINDINGS · 6 ANALYSIS AREAS · F-ID TICKET SYSTEM**

---


## Inhalt

- [Structured Findings](#structured-findings)
- [Findings](#findings)


### Structured Findings

Jedes Finding bekommt eine eindeutige ID wie `F-NET-07` oder `F-WEAT-01`. Das ist kein Formalismus — es löst ein konkretes Problem in längeren Analyseprojekten.

**Das Problem ohne IDs:** Erkenntnisse aus Notebook A beeinflussen Entscheidungen in Notebook C. Ohne Referenz sieht man das Ergebnis, aber nicht die Begründung. Nach zwei Wochen weiß man nicht mehr warum ein Feature gewählt wurde.

**Mit Finding-IDs entsteht eine lückenlose Kette:**

```
F-NET-07 (03_analysis_2-network.ipynb)
    "Pearson r ≥ 0.85 zwischen aufeinanderfolgenden Delays eines Trips"
    Impact: High · Action: → Feature Engineering
        ↓
prev_trip_delay (05_feature_engineering.ipynb)
    "Kaskadenindikator via trip_id — direkt aus F-NET-07"
        ↓
LightGBM v2 MAE: 18,56 s (06_prediction_4-model_v2.ipynb)
    Feature Importance Rank #2 → bestätigt F-NET-07
```

Das entspricht professionellen Daten-Team-Workflows — analog zu Ticket-Systemen wie Jira. Jede Feature-Entscheidung ist auf eine empirische Evidenz zurückführbar, nicht auf Intuition.

> **Claim:** "Analysis dictates the model — no feature was added speculatively."
> Diese ID-Struktur ist der Nachweis dafür.

## Findings


> Alle Findings werden in den jeweiligen Analysis-Notebooks erarbeitet und hier zentral gelistet.
> ID-Schema: `F-{NOTEBOOK}-{NR}` — Status: `open` · `in-progress` · `done`
> Präsentation: `hot` = starkes Einzel-Finding · `story` = Teil einer größeren Erzählung · `—` = technisch/intern

| # | ID | Notebook | Section | Finding | Präs. | Impact | Action | Status |
|---:|:---|:---|:---|:---|:---|:---|:---|:---|
| 1 | F-TARGET-01 | target | Distribution | `arrival_delay` rechtsschiefe Verteilung — Median 42s vs. Mean 56.3s | `—` | Lineare Modelle unterschätzen Extremwerte | Log-Transform + MdAE-Check | done |
| 2 | F-TARGET-02 | target | Delay Delta | `delay_delta` bimodal — Recovery-Cluster ~−45s und Akkumulations-Cluster ~+15s; kein Datenfehler | `story` | Systematisches Signal im delta | `delay_delta` als Nebenmetrik | done |
| 3 | F-TARGET-03 | target | Delay Delta | **71.5% `delay_delta > 0`** — kein Fahrplanpuffer; Trams akkumulieren an über 2/3 aller Halte | `hot` | Systemischer Puffermangel | `delay_delta` als Nebenmetrik | done |
| 4 | F-TARGET-04 | target | Target | `dwell_time` verfügbar; 71.3% = 0s (Durchfahrten ohne Haltezeit) | `—` | Feature schwächer als erwartet | `has_dwell` als Binary | done |
| 5 | F-TARGET-05 | target | Cancellations | `canceled`-Artefakt vor Jul 2024 — kein reales Betriebsproblem. Effektive Rate: 6.2% | `—` | Verfälscht Cancellation-Baseline | `canceled=True` aus Delay-Modell | done |
| 6 | F-TARGET-06 | target | Monthly Trend | Nov–Dez 2025 GTFS-Artefakt: delay_delta springt auf +17.1s/+26.0s — kein echter Betriebstrend | `—` | Verfälscht Trendanalyse | Nov–Dez 2025 aus Train+Test entfernen | done |
| 7 | F-TARGET-07 | target | Extremwerte | Extremwerte bis ±3600s — wahrscheinlich echte Grossstörungen, kein Messfehler | `—` | Robust-Modell bevorzugen | Robust-Loss (MdAE + Huber) | done |
| 8 | F-TARGET-08 | target | OTP | `trip_id` + `stop_sequence` im Master-Datensatz — Kaskadenanalyse möglich | `—` | Prediction-Signal verfügbar | `trip_id` in Feature Engineering | done |
| 9 | F-TARGET-09 | target | Trend | delay_delta Trend: j23=+4.6s → j24=+5.1s → j25=+5.1s (moderat, stabil) | `story` | Struktureller Aufwärtstrend real aber nicht alarmierend | `year` + `month` als Features | done |
| 10 | F-TARGET-10 | target | Trend | `arrival_delay` 2025 (Jan–Okt): **55.8s** — leicht unter 2024 (59.4s) → Stabilisierung | `story` | Netz wird nicht in allen Metriken schlechter | In Trend-Analyse hervorheben | done |
| 11 | F-TARGET-11 | target | Cancellations | Synchrone Cancellation-Erhöhung aller Linien vor Jul 2024 — beweist Datendefinitions-Änderung | `—` | Stärkstes Qualitäts-Argument | `is_pre_july_2024` als Feature | done |
| 12 | F-TARGET-12 | target | Outlier | **Linie E**: OTP 55.7%, Ø 130s, 2'511 Zeilen — aus lf_clean entfernt | `—` | Outlier verzerrt Modell-Baseline | Als Sonderlinie annotieren | done |
| 13 | F-TARGET-13 | target | Datenstrategie | Bereinigung minimal (+1.1s arr, −0.8s delta) — Tendenzen unverändert | `—` | lf_clean ist saubere Modellbasis | lf_clean als Standard | done |
| 14 | F-NET-01 | network | Netzveränderungen | L9/L11/L13 Dez 2023: +8/+13/+19 Halte im GTFS. Teils Artefakt (Innenstadtachse), teils echte Erweiterungen (L13→Sihlcity, L11→Rehalp) | `story` | Jahresvergleiche nur mit Kontext | `gtfs_year` Feature kodiert Zeitschnitt | done |
| 15 | F-NET-02 | network | Netzveränderungen | Stabile Referenzlinien: L10, L12, L14, L17 identisch über alle Jahre | `—` | Kontrollgruppe für Zeitreihen | Als Referenzlinien verwenden | done |
| 16 | F-NET-03 | network | Feature | `gtfs_year` erklärt netzweit nur +0.5s — schwaches Feature | `—` | Kein klarer Netzwechsel-Effekt | `n_stops_line` als Alternative | done |
| 17 | F-NET-04 | network | Einlaufzeit | Kein Einlaufzeit-Effekt: Unterschiede lagebezogen, nicht zeitbezogen | `—` | Neubaustrecken keine Einlauf-Toleranz nötig | `is_new_stop` wenig aussagekräftig | done |
| 18 | F-NET-05 | network | Hotspots | **Keine Korrelation** Linienanzahl × Delay: Central/Paradeplatz (je 7 Linien, ~49s) — beide unter Netzschnitt | `hot` | Kaskadenrisiko-Hypothese widerlegt | Aussenkorridore statt Knotenpunkte | done |
| 19 | F-NET-06 | network | Versorgung | Kreis 12 (+2) und Kreis 4 (+2) gewinnen Linien. Kreis 7 verliert 2 Linien | `—` | Räumliche Netzstruktur-Änderung | Kreise 12/4 verbessert; Kreis 7 verschlechtert | done |
| 20 | F-NET-07 | network | Kaskaden | `trip_id` ermöglicht `prev_trip_delay` als Feature — **implementiert als stärkstes Feature in v2** | `—` | F-REC-01 | `06_prediction_4` | done |
| 21 | F-NET-08 | network | Linie E | Linie E: 130s Ø Delay — Entlastungslinie, separat behandeln | `—` | Extremer Outlier | Als Sonderlinie annotieren | done |
| 22 | F-NET-09 | network | Hotspots | **Netzausbau vs. Delay-Hotspots — kein Overlap:** Echte Erweiterungen in K3/K8 (gut performend). K11/K12 (Problemkreise) erhielten nichts | `hot` | Netz ausgebaut, aber nicht wo es gebraucht wird | Kontext für VBZ-Empfehlung | done |
| 23 | F-TEMP-01 | temporal | Stunden | Kein Morgenrush (7h=48.9s unter Ø). Peak **21h=67.9s** (Events-Abreisewelle), 17h=65.2s | `hot` | `hour` stärkstes temporales Feature | `hour` + `hour × has_event` | done |
| 24 | F-TEMP-02 | temporal | Wochentag | **Donnerstag** kritischster Tag: Ø 60.4s, P95=194s. Montag (52.3s) und Sonntag (48.4s) beste Tage | `hot` | `weekday` als Feature | `day_of_week` ordinalkodiert | done |
| 25 | F-TEMP-03 | temporal | Wochentag | Donnerstag-Peak: Events-Häufung (Do-Abend) + HO-Hypothese — nicht direkt belegt | `story` | Interaktion Do × Abend × Events | `is_school_week` als Modifier | done |
| 26 | F-TEMP-04 | temporal | Wochentag | Samstag (57.0s) kaum besser als Werktag; Sonntag (48.4s) deutlich besser | `—` | `weekday` ordinalkodiert statt binär | `day_of_week` 7-Kategorien | done |
| 27 | F-TEMP-05 | temporal | Monat | **November-Peak**: Nov 2023=68.9s, Nov 2024=72.6s — jeweils Jahreshöchstwert | `hot` | `is_november` als Feature-Flag | Ursache: Laub + Baustellensaison + MIV | done |
| 28 | F-TEMP-06 | temporal | Saison | Herbst=61.2s schlechteste; **Winter=51.7s beste Jahreszeit** (OTP 88.9%) — kontraintuitiv | `story` | `season` als Feature | Winter-Vorteil: MIV-Reduktion | done |
| 29 | F-TEMP-07 | temporal | Zeitreihe | Aufwärtstrend: 2024 +4–7s über 2023; 2025 leicht moderater (Stabilisierung) | `story` | `year` + `month` als Features | Rolling-Baseline als Feature-Idee | done |
| 30 | F-TEMP-08 | temporal | Zeitreihe | Schulferien-Täler im Rolling-Average erkennbar | `—` | Ferieneffekt additiv | `is_school_holiday` aus ZH-Kalender | done |
| 31 | F-TEMP-09 | temporal | Feature | `gtfs_year` netzweit +0.5s — bestätigt F-NET-03 | `—` | Schwaches Feature | Empirisch evaluieren | done |
| 32 | F-TEMP-10 | temporal | Stunden | Nacht-/Partyverkehr (0–3h): Fr/Sa leichter 2h-Anstieg — Partygänger-Rückfahrten (n datendünn) | `—` | `hour × is_weekend` als Interaktion | In Feature Engineering | done |
| 33 | F-SPAT-01 | spatial | Hotspots | Hotspots sind periphere Aussenkorridore: Friedhof Enzenbühl 93.8s, Balgrist 85.2s — **nicht** zentrale Knotenpunkte | `hot` | `stop_name` stärkster räumlicher Prädiktor | Target-Encoding + n-Threshold | done |
| 34 | F-SPAT-02 | spatial | Terminus | Terminus-Frühankünfte: lf_all=55.8s vs. lf_clean=56.9s (Δ nur 1s) | `—` | Kein Verzerrungseffekt | n-Threshold-Filter | done |
| 35 | F-SPAT-03 | spatial | Stadtkreise | **Kreis 11** schlechtester (68,3 s, OTP 83%), Kreis 12 (66.3s). Kreis 5 bester (49.9s, OTP 89%) | `hot` | `district_nr` additiv nützlich | K11/K12 als High-Risk-Marker | done |
| 36 | F-SPAT-04 | spatial | Linien | Alle Linien akkumulieren (delta > 0). Stärkste: L4 (+8.1s), L10 (+6.5s), L11 (+6.2s) | `story` | `line_name` als Feature | Linien-Encoding | done |
| 37 | F-SPAT-05 | spatial | Linien | `line_name` stärkster räumlicher Prädiktor. L11 (68.7s, OTP 82%) kritischste Hauptlinie | `—` | Beide Features ins Modell | Target-Encoding für `stop_name` | done |
| 38 | F-SPAT-06 | spatial | Starthalte | Starthaltestellen-Proxy: 0 Kandidaten — Verzerrung 0.0s | `—` | n-Threshold-Filter empfohlen | `n_threshold` in Preprocessing | done |
| 39 | F-SPAT-07 | spatial | Linien-Dichte | **0 Overlap** Top-20-Linienanzahl × Top-20-Delay. Haldenegg (15 Linien, 44.5s), Paradeplatz (14 Linien, 48.2s) — alle unter Netzschnitt | `story` | `n_lines_at_stop` schwaches Feature | Aussenkorridore statt Knotenpunkte | done |
| 40 | F-SPAT-08 | spatial | dwell_time | `dwell_time` = 0s für **71.3%** — kein Puffer eingebaut. System akkumuliert unweigerlich | `hot` | Feature schwächer als erwartet | `has_dwell` testen | done |
| 41 | F-SPAT-09 | spatial | Endstationen | **Endstationen-Muster:** L11/L13/L7 grosse Delay-Bubbles an Start und Ende. Linienlänge als Proxy-Feature prüfen | `story` | Peripheral-Effekt messbar | `n_stops_line` als Feature | done |
| 42 | F-SPAT-10 | spatial | Richtung | Fahrt Richtung Aussenquartiere akkumuliert mehr Delay als Rückfahrt. `trip_direction` als Feature prüfen | `—` | Asymmetrie messbar | `trip_direction` (letzter Stop) | done |
| 43 | F-SPAT-11 | spatial | Heatmap | Abend-Peak (17–19 Uhr) netzweit synchron. L11/L8 hohes Grundniveau ganztags | `—` | `hour × line_name` Interaktion stärker als Einzelfeatures | In Feature Engineering | done |
| 44 | F-WEAT-01 | weather | Schnee | **Schnee stärkster Wettereffekt**: +54.0s, OTP 87.1%→76.1% (−10.9pp) | `hot` | `has_snow` wichtigstes Wetter-Feature | `has_snow` priorisieren | done |
| 45 | F-WEAT-02 | weather | Regen | Starkregen: +23.3s. Dosis-Wirkungs: <2mm=62.6s → >10mm=89.5s | `story` | `precipitation` kontinuierlich als Feature | `rain_scale` 0–4 | done |
| 46 | F-WEAT-03 | weather | Wind | `is_windy` = NaN — nie befüllt. **Aus Feature-Set entfernt.** | `—` | Feature-Set bereinigt | `is_windy` entfernt | done |
| 47 | F-WEAT-04 | weather | Temperatur | 0–5°C = bester Bereich (53.8s). `is_hot` (>20°C) = +2.0s — schwaches Signal | `—` | Frost-Hypothese falsch | `temperature` kontinuierlich | done |
| 48 | F-WEAT-05 | weather | Korrelation | Alle Wetter-Features schwach (max r=0.042) — unabhängige Signale | `—` | Kein Multikollinearitätsproblem | Alle behalten | done |
| 49 | F-WEAT-06 | weather | Features | `precipitation` (r=0.036) und `has_snow` (r=0.038) nützlichste Features | `—` | Priorität: Schnee + Niederschlag | Feature-Selection nach Training | done |
| 50 | F-WEAT-07 | weather | Geografie | **Geografische Trennung:** Schnee trifft Höhenlagen (K10/K4/K12), Regen trifft Flusstäler (K5). Bahnhof Selnau Schnee-Extremausreisser: +190.9s | `hot` | Topographie bestimmt Vulnerabilität | `district × has_snow` als Interaktion | done |
| 51 | F-WEAT-08 | weather | Regen | **Regen-Korridor K5:** 11/20 Top-Regen-Halte im Escher Wyss / Toni-Areal / Limmat. Toni-Areal +44.2s | `story` | Geografisch konzentrierter Effekt | K5 × `has_heavy_rain` | done |
| 52 | F-WEAT-09 | weather | Linien | **Linien reagieren komplett unterschiedlich:** L17 Schnee +7.7s vs. Regen +41.2s (Limmat-Route). L9 Schnee +75.9s vs. Regen +10.0s (Höhenlagen) | `hot` | `line × Wettertyp` Interaktionsterm | `line_name × has_snow` | done |
| 53 | F-EVNT-01 | events | Feiertage | Feiertage **46.3s vs. Normal 56.2s (−9.9s, OTP +3.6pp)** — bester Tagestyp | `hot` | `is_holiday` wichtigstes Event-Feature | `is_holiday` in `02_preparation` | done |
| 54 | F-EVNT-02 | events | Event-Grösse | Gross=66.7s (+10.5s), Mittel=58.9s (+2.7s), Klein=56.2s (+0.05s ≈ Normal) | `story` | `event_weight` ordinal; Klasse 1 binarisieren | `event_weight ≥ 2` als Schwelle | done |
| 55 | F-EVNT-03 | events | Stunden | Event-Effekt primär **Abend-Phänomen (18–22h)** — tagsüber kein Unterschied. Erklärt 21h-Spike | `hot` | `has_event × hour` Interaktion | `event_weight × hour` | done |
| 56 | F-EVNT-04 | events | Event-Typ | **Fachmessen schlechteste Kategorie** (66.0s, OTP 84%) — nicht Konzerte. Super League 53.8s ≈ Normal | `story` | `event_type` kategorisch | Fachmessen-Effekt für L11 | done |
| 57 | F-EVNT-05 | events | Balance | Gross-Events n=724k vs. Normal 70.5M — stark unbalanced | `—` | Oversampling oder gewichtetes Training | Event-Strategie in Modell-Phase | done |
| 58 | F-EVNT-06 | events | Stadtkreise | Stadtkreis-Δ auf Event-Tagen minimal (max +3.0s K2). Räumliche Aggregation verbirgt Abend-Effekt | `—` | `has_event × hour` aussagekräftiger | Abend-Fokus in Feature Engineering | done |
| 59 | F-SIM-01 | sim | dwell_time | `dwell_time` ist Feature #1 in lgbm_v1 (Gain 14.8M > stop_name 12.7M), aber **faktisch binär**: 0s (71.3%) oder 60s (28.5%) — Werte 1–59s existieren nicht im VBZ-Fahrplan | `hot` | Binäre Verteilung limitiert direkte Simulation — kein kontinuierlicher Hebel | Stopspezifische Kalibrierung statt pauschaler 0/60 | done |
| 60 | F-SIM-02 | sim | dwell_time | `dwell_time` korreliert **positiv** mit Delay (r=+0.16): Stops mit 60s haben ~28s mehr Delay. Ursache: Konfundierung — VBZ gibt Puffer an komplexen Stops, die strukturell mehr Delay haben | `hot` | Feature Importance ≠ kausaler Hebel; Modell hat Korrelation korrekt gelernt | Konfundierung explizit kommunizieren | done |
| 61 | F-SIM-03 | sim | dwell_time | Simulation 0→60s: Modell erhöht Vorhersage um +20s (L11: +19.96s, netzweit: +20.72s). Modell kann nicht unterscheiden ob dwell_time=60 wegen Stopschwierigkeit (historisch) oder als Puffer (hypothetisch) | `story` | Observational ML isoliert Kausaleffekt nicht | A/B-Test oder Instrumental Variable für Validierung | done |
| 62 | F-SIM-04 | sim | dwell_time | Operative Empfehlung trotzdem valide (F-SPAT-08 + Domänenwissen): stopspezifische dwell_time statt 0/60. Quantifizierung erfordert A/B-Test. Modell liefert Diagnose, Betrieb liefert Dosis | `story` | Kausalinferenz braucht experimentelle Daten | Pilot-Experiment mit ausgewählten Stops | done |
| 63 | F-REC-01 | rec | full-circle | **Vorhersagbar = Strukturell = Steuerbar.** MAE 18,56 s beweist: Delays folgen Mustern. Zufällige Delays wären nicht so präzise vorhersagbar. Was Muster hat, kann durch Fahrplandesign beeinflusst werden | `hot` | Modell transformiert Analyse-Findings in operative Handlungsgrundlage | Modell-Output als Input für Schedule-Design verwenden | done |
| 64 | F-REC-02 | rec | recommendations | Risiko-Matrix (Stop × Linie × Kontext) aus lgbm_v1-Vorhersagen identifiziert Stop-Linie-Kontext-Kombinationen mit pred. Delay >60s — Grundlage für stopspezifische dwell_time-Kalibrierung | `hot` | Präzisere Pufferstrategie als pauschale 0/60s | Empfehlungstabelle als Fahrplan-Input nutzen | done |
| 65 | F-REC-03 | rec | recommendations | Kontextspezifische Muster: Schnee betrifft andere Stops als Events oder Rush-Hour. Einheitliche Pufferstrategie greift zu kurz → kontextsensitive Fahrpläne (Schneefahrplan, Eventfahrplan) sind die logische Konsequenz | `story` | Mehrere Betriebs-Szenarien mit je eigenem Puffer-Raster | Kontextsensitive Fahrpläne entwerfen | done |
| 66 | F-REC-04 | rec | recommendations | Empfohlene Puffergrößen sind Startpunkte (Heuristik: 1/3 Überschuss, gerundet auf 5s). Validierung durch operatives A/B-Testing: Modell liefert Diagnose, Betrieb liefert Dosis | `story` | Kausale Wirkung nicht aus Observational-Daten ableitbar | Randomisiertes Pilot-Experiment für Kalibrierung | done |
